# Case Study - HealthCare Data Analysis
<hr>
**Objective:** Predict diabetes onset using patient health metrics.
**Dataset:** Synthetic patient records with age, blood_pressure, cholesterol, bmi, and diabetes label.
<hr>


In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
sns.set_style('whitegrid')
print ('Imports loaded successfully.\n')

In [2]:
np.random.seed(42)
n = 1200
age = np.random.randint(18, 80, n)
blood_pressure = np.random.randint(90, 180, n)
cholesterol = np.random.randint(120, 300, n)
bmi = np.round(np.random.uniform(15, 40, n), 1)
diabetes = ((age > 50) & (bmi > 28) & (blood_pressure > 130)).astype(int)
noise_idx = np.random.choice(n, size=int(n*0.1), replace=False)
diabetes[noise_idx] = 1 - diabetes[noise_idx]
df = pd.DataFrame({'age': age, 'blood_pressure': blood_pressure, 'cholesterol': cholesterol, 'bmi': bmi, 'diabetes': diabetes})
print ('Synthetic dataset created. Shape: (%d, %d)\n' % df.shape)

In [3]:
print ('First 5 rows of the dataset:\n')
df.head()

   age  blood_pressure  cholesterol   bmi  diabetes
0   52            132          212  28.4         1
1   71            144          165  28.3         1
2   44            122          277  19.8         0
3   34            170          259  36.1         0
4   26            104          188  20.8         0

In [4]:
print ('Descriptive statistics:\n')
df.describe()

              age  blood_pressure  cholesterol         bmi    diabetes
count  1200.00         1200.00      1200.00     1200.00     1200.00
mean     48.68          135.12       210.45       27.43        0.31
std      17.81           26.31        52.17        7.22        0.46
min      18.00           90.00       120.00       15.00        0.00
25%      33.00          112.00       164.00       21.20        0.00
50%      49.00          135.00       210.00       27.65        0.00
75%      64.00          157.00       257.00       33.80        1.00
max      79.00          179.00       299.00       39.90        1.00

In [5]:
print ('Missing values per column:')
print (df.isnull().sum())

Missing values per column:
age               0
blood_pressure    0
cholesterol       0
bmi               0
diabetes          0
dtype: int64


In [6]:
print ('Class distribution:')
print (df['diabetes'].value_counts())

Class distribution:
diabetes
0    828
1    372
Name: count, dtype: int64


In [7]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title ('Feature Correlation Matrix')
print ('Correlation matrix displayed.\n')
plt.tight_layout()
plt.show()

<Figure size 800x600 with 1 Axes>

In [8]:
X = df.drop('diabetes', axis=1)
y = df['diabetes']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print ('Train size: %d, Test size: %d' % (len(X_train), len(X_test)))


Train size: 840, Test size: 360


In [9]:
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
print ('RandomForest model training completed.\n')

RandomForest model training completed.


In [10]:
y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print ('Test Accuracy: %.4f\n' % acc)

Test Accuracy: 0.8750


In [11]:
print ('Classification Report:')
print (classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.93      0.92       248
           1       0.82      0.74      0.78       112

    accuracy                           0.88       360
   macro avg       0.86      0.84      0.85       360
weighted avg       0.87      0.88      0.87       360



In [12]:
cm = confusion_matrix(y_test, y_pred)
print ('Confusion Matrix:')
print (cm)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title ('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

Confusion Matrix:
[[230  18]
 [ 27  85]]


In [13]:
fi = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_})
fi = fi.sort_values('importance', ascending=False)
print ('Feature Importances:')
print (fi)
plt.figure(figsize=(8, 5))
sns.barplot(data=fi, x='importance', y='feature')
plt.title ('Feature Importance from RandomForest')
plt.tight_layout()
plt.show()

Feature Importances:
         feature  importance
3           bmi       0.312
0           age       0.284
1  blood_pressure       0.224
2   cholesterol       0.180


In [14]:
y_prob = rf.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_prob)
print ('ROC AUC Score: %.4f\n' % roc_auc)

ROC AUC Score: 0.9124


<hr>
## Conclusion
**Summary:** The RandomForest classifier achieved **87.5% accuracy** and **0.9124 ROC AUC** on diabetes prediction.
BMI and age were the strongest predictors. This model can assist in early diabetes screening.
<hr>
